# 15.4 - Privacy & Data Protection

**Module 15 - Ethics & Responsible AI** | Netsetos GenAI Engineering

This notebook works through Privacy & Data Protection hands-on: Classify the personal data you hold; Measure how re-identifiable the 'anonymous' data is; Hide each record in a crowd with k-anonymity; Close the k-anonymity hole with l-diversity; Buy the guarantee that survives any side data: differential privacy; Test whether a trained model leaks who was in it.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A one-line reproducibility note. Lesson 15.4 is deliberately keyless and dependency-free: every block is plain-Python arithmetic on one small illustrative clinic dataset, so there is nothing to install and nothing to import here.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A comment-only setup cell, not real code. It flags that the lesson uses fixed, seeded values so every run prints the same numbers - which matters because a few blocks stand in a single 'drawn' random sample for a real random draw (the differential-privacy noise) to keep outputs reproducible.

**How the code works - step by step**
- **The comment** - notes that seeded / fixed random values are used throughout so results are deterministic.
- No imports, no state - the only later import is `math` inside the differential-privacy cell.

**In one line:** nothing to install; the whole lesson is keyless arithmetic on one toy dataset.

**What you'll see:** (no output - environment prep)

## 1 - Classify the personal data you hold

Privacy work starts by finding the personal data and sorting it into three classes: **direct** identifiers (name, email, SSN) that name a person on their own, **quasi**-identifiers (ZIP, birth date, sex) that name them only in combination, and the **sensitive** attribute (a diagnosis) that an attacker is trying to learn. You cannot protect what you have not classified, and deleting the direct identifiers is where most teams stop - which the next step shows is not enough.

In [ ]:
# PII DETECTION & CLASSIFICATION: privacy work starts by finding the personal data. Fields split into DIRECT identifiers
# (identify a person alone - name, email, SSN) and QUASI-identifiers (identify only in COMBINATION - ZIP, birth date, sex).
rows = [  # (name, email, zip, birth, sex, diagnosis) - an illustrative clinic export
    ("Ada",   "ada@x.com",   "02138", "1985-03-11", "F", "flu"),
    ("Bao",   "bao@x.com",   "02139", "1985-07-22", "M", "flu"),
    ("Chen",  "chen@x.com",  "02141", "1985-09-02", "F", "flu"),
    ("Diego", "diego@x.com", "02142", "1985-01-30", "M", "flu"),
    ("Eve",   "eve@x.com",   "10001", "1972-06-15", "F", "diabetes"),
    ("Farah", "farah@x.com", "10002", "1972-11-03", "F", "cancer"),
    ("Gus",   "gus@x.com",   "10003", "1972-02-19", "M", "asthma"),
    ("Hana",  "hana@x.com",  "10004", "1972-08-27", "F", "flu")]
fields = ["name", "email", "zip", "birth", "sex", "diagnosis"]
DIRECT = {"name", "email"}            # each identifies a person on its own
QUASI = {"zip", "birth", "sex"}       # identify a person only in combination
SENSITIVE = {"diagnosis"}             # what an attacker wants to learn
print("{} records, {} fields.".format(len(rows), len(fields)))
print("  direct identifiers   : {}".format(sorted(DIRECT)))
print("  quasi-identifiers    : {}".format(sorted(QUASI)))
print("  sensitive attribute  : {}".format(sorted(SENSITIVE)))
with_direct = sum(1 for r in rows if r[0] or r[1])
print()
print("{}/{} records carry a DIRECT identifier - dropping name+email is necessary but NOT sufficient.".format(with_direct, len(rows)))
print("The quasi-identifiers still leak: the next step shows they can re-identify a person all by themselves.")

# Output:
# 8 records, 6 fields.
#   direct identifiers   : ['email', 'name']
#   quasi-identifiers    : ['birth', 'sex', 'zip']
#   sensitive attribute  : ['diagnosis']
#
# 8/8 records carry a DIRECT identifier - dropping name+email is necessary but NOT sufficient.
# The quasi-identifiers still leak: the next step shows they can re-identify a person all by themselves.

**Explanation**

A classification-and-count harness over an eight-record clinic export, not a model call. It labels each field as DIRECT, QUASI, or SENSITIVE with three Python sets, then counts how many records carry a direct identifier to make the point that removing name and email is necessary but nowhere near sufficient.

**How the code works - step by step**
- **`rows`** - eight illustrative clinic records, each a tuple `(name, email, zip, birth, sex, diagnosis)`.
- **`DIRECT` / `QUASI` / `SENSITIVE`** - three sets that assign every field to a class: name+email identify alone, zip+birth+sex identify only in combination, diagnosis is the secret.
- **The class printout** - reports the counts and the sorted field names in each class.
- **`with_direct`** - counts records where a name or email is present (all 8), then prints that dropping them is necessary but not sufficient because the quasi-identifiers remain.

**In one line:** sort every field into direct / quasi / sensitive, then show all 8 rows still carry a direct identifier - the obvious PII to remove first.

**What you'll see:** Prints '8 records, 6 fields', the three field classes, then '8/8 records carry a DIRECT identifier - dropping name+email is necessary but NOT sufficient' with a pointer to the re-identification step.

## 2 - Measure how re-identifiable the 'anonymous' data is

Strip the names and the data still is not anonymous. Latanya Sweeney's landmark result is that (5-digit ZIP + full birth date + sex) uniquely identifies about **87 percent** of the US population - she re-identified a governor's medical record by cross-linking an 'anonymous' hospital release to a public voter roll. A record that is unique on its quasi-identifiers is directly re-identifiable.

In [ ]:
# RE-IDENTIFICATION VIA QUASI-IDENTIFIERS: strip the names and the data still is not anonymous. A famous result (Sweeney)
# is that {5-digit ZIP, full birth date, sex} uniquely identifies ~87% of the US population. Count how many rows are unique.
rows = [  # (zip, birth, sex) - the quasi-identifiers that survive after name+email are removed
    ("02138", "1985-03-11", "F"), ("02139", "1985-07-22", "M"), ("02141", "1985-09-02", "F"),
    ("02142", "1985-01-30", "M"), ("10001", "1972-06-15", "F"), ("10002", "1972-11-03", "F"),
    ("10003", "1972-02-19", "M"), ("10004", "1972-08-27", "F")]
counts = {}
for q in rows:
    counts[q] = counts.get(q, 0) + 1
unique = [q for q in rows if counts[q] == 1]
print("Names removed. Re-identification risk from the quasi-identifiers (zip, birth, sex):")
print("  {} of {} records are UNIQUE on their quasi-identifiers -> {:.0%} directly re-identifiable.".format(len(unique), len(rows), len(unique) / len(rows)))
print()
print("Anonymity is not just 'delete the name'. Sweeney showed {ZIP, birth date, sex} pins down ~87% of Americans by cross-linking")
print("to a public voter roll. The fix is to make each record blend into a crowd - which is what k-anonymity measures next.")

# Output:
# Names removed. Re-identification risk from the quasi-identifiers (zip, birth, sex):
#   8 of 8 records are UNIQUE on their quasi-identifiers -> 100% directly re-identifiable.
#
# Anonymity is not just 'delete the name'. Sweeney showed {ZIP, birth date, sex} pins down ~87% of Americans by cross-linking
# to a public voter roll. The fix is to make each record blend into a crowd - which is what k-anonymity measures next.

**Explanation**

A measurement harness that quantifies re-identification risk on the quasi-identifiers alone (name and email already removed). It counts how many of the eight records are unique on their `(zip, birth, sex)` tuple and reports that as the fraction directly re-identifiable - the concrete version of Sweeney's 87 percent.

**How the code works - step by step**
- **`rows`** - the eight records reduced to just their surviving quasi-identifiers `(zip, birth, sex)`.
- **`counts`** - a dictionary tallying how many records share each exact quasi-identifier tuple.
- **`unique`** - the records whose tuple appears exactly once (count == 1).
- **The printout** - reports unique-count / total as a percentage and explains the attack: cross-link the 'anonymous' table to a public voter roll on the shared quasi-identifiers.

**In one line:** count rows unique on their quasi-identifiers - here every one - to show 'delete the name' is not anonymity.

**What you'll see:** Prints that 8 of 8 records are unique on their quasi-identifiers -> 100% directly re-identifiable, followed by the Sweeney cross-link explanation and a pointer to k-anonymity as the fix.

## 3 - Hide each record in a crowd with k-anonymity

If uniqueness is the problem, the fix is to make each record blend in. A table is **k-anonymous** if every record shares its quasi-identifiers with at least k-1 others, so an attacker who narrows down to a record narrows down to a group of at least k people. You raise k by **generalizing** (ZIP 02138 -> 02, a birth year -> a decade) and **suppressing** (dropping a column) - which buys privacy at the cost of coarser data.

In [ ]:
# k-ANONYMITY: a table is k-anonymous if every record shares its quasi-identifiers with at least k-1 others - so no one is
# unique. You raise k by GENERALIZING (zip 02138 -> 02, birth 1985 -> the 1980s) and SUPPRESSING (drop sex). Measure k.
raw = [("02138", "1985", "F"), ("02139", "1985", "M"), ("02141", "1985", "F"), ("02142", "1985", "M"),
       ("10001", "1972", "F"), ("10002", "1972", "F"), ("10003", "1972", "M"), ("10004", "1972", "F")]
def k_of(table):
    groups = {}
    for q in table:
        groups[q] = groups.get(q, 0) + 1
    return min(groups.values()), groups
def generalize(z, y, s):
    return (z[:2], y[:3] + "0s")            # zip -> 2-digit region, year -> decade, sex SUPPRESSED (dropped)
k_raw, _ = k_of(raw)
gen = [generalize(*r) for r in raw]
k_gen, groups = k_of(gen)
print("raw quasi-identifiers (zip5, year, sex): k = {}  (every record is unique -> re-identifiable)".format(k_raw))
print("after generalize+suppress (zip2, decade, drop sex):")
for g, n in sorted(groups.items()):
    print("  {} -> {} records".format(g, n))
print("  k = {}  (each record now hides among {} identical ones)".format(k_gen, k_gen))
print()
print("k-anonymity trades detail for privacy: bigger k is more private but coarser data. But k alone is not enough -")
print("if everyone in a group shares the same secret, hiding WHICH person you are does not hide WHAT you have. See l-diversity.")

# Output:
# raw quasi-identifiers (zip5, year, sex): k = 1  (every record is unique -> re-identifiable)
# after generalize+suppress (zip2, decade, drop sex):
#   ('02', '1980s') -> 4 records
#   ('10', '1970s') -> 4 records
#   k = 4  (each record now hides among 4 identical ones)
#
# k-anonymity trades detail for privacy: bigger k is more private but coarser data. But k alone is not enough -
# if everyone in a group shares the same secret, hiding WHICH person you are does not hide WHAT you have. See l-diversity.

**Explanation**

A before/after measurement of k over the same records. It computes k on the raw quasi-identifiers, then applies generalization and suppression and recomputes k, making the privacy-utility tradeoff visible: k rises from 1 to 4 while the data gets coarser.

**How the code works - step by step**
- **`raw`** - the eight quasi-identifier tuples with the birth date already reduced to a year.
- **`k_of`** - groups records by identical quasi-identifiers and returns the smallest group size (that is k) plus the group counts.
- **`generalize`** - coarsens a record: ZIP to its first two digits, year to a decade string, and sex suppressed (dropped entirely).
- **`k_raw` vs `k_gen`** - k of the raw table (1, everyone unique) versus k after generalizing and suppressing.
- **The printout** - shows the two resulting equivalence classes and the new k, then names the tradeoff and previews l-diversity's gap.

**In one line:** generalize + suppress until the smallest quasi-identifier group has k members - raw k=1 becomes k=4.

**What you'll see:** Prints raw k=1, then the two generalized groups ('02','1980s') and ('10','1970s') at 4 records each -> k=4, with a note that bigger k is more private but coarser and that k alone is not enough.

## 4 - Close the k-anonymity hole with l-diversity

k-anonymity guarantees you cannot tell **which** of the k people a record is, but says nothing about **what** they share. If every member of a k-anonymous group has the same sensitive value, membership alone discloses it - the **homogeneity attack**. l-diversity requires at least l distinct sensitive values per group; t-closeness goes further and matches the group's distribution to the whole dataset.

In [ ]:
# l-DIVERSITY (the k-anonymity homogeneity attack): a group can be k-anonymous yet still leak. If all k members share the
# same SENSITIVE value, learning someone is in the group reveals their value even though you cannot tell which person they are.
groups = {   # each is a k=4 equivalence class after generalization: (quasi-class -> list of sensitive values)
    "(02, 1980s)": ["flu", "flu", "flu", "flu"],                 # homogeneous
    "(10, 1970s)": ["diabetes", "cancer", "asthma", "flu"]}      # diverse
L_MIN = 2                                                        # require at least 2 distinct sensitive values per group
for cls, vals in groups.items():
    l = len(set(vals))
    ok = l >= L_MIN
    verdict = "OK (l-diverse)" if ok else "LEAKS - homogeneity attack (every member has the same value)"
    print("class {}: k=4, distinct sensitive values l={} -> {}".format(cls, l, verdict))
print()
print("Both groups are k-anonymous (k=4), but the first is l=1: anyone known to be a 40-something in the 02 region HAS the flu,")
print("no name needed. l-diversity requires variety in the sensitive column; t-closeness goes further and matches its distribution.")

# Output:
# class (02, 1980s): k=4, distinct sensitive values l=1 -> LEAKS - homogeneity attack (every member has the same value)
# class (10, 1970s): k=4, distinct sensitive values l=4 -> OK (l-diverse)
#
# Both groups are k-anonymous (k=4), but the first is l=1: anyone known to be a 40-something in the 02 region HAS the flu,
# no name needed. l-diversity requires variety in the sensitive column; t-closeness goes further and matches its distribution.

**Explanation**

A check harness over two k=4 equivalence classes, contrasting a leaky group with a safe one. It counts distinct sensitive values per group and flags any group below the l threshold, demonstrating that k-anonymity and l-diversity protect different things - identity versus the secret.

**How the code works - step by step**
- **`groups`** - two k=4 classes mapping a quasi-identifier class to its list of sensitive diagnoses; one homogeneous, one diverse.
- **`L_MIN`** - the diversity requirement: at least 2 distinct sensitive values per group.
- **The loop** - for each class computes `l = len(set(vals))` and marks it OK if `l >= L_MIN`, else flags the homogeneity leak.
- **The printout** - reports each class's l and verdict, then explains that a 40-something in the '02' region provably has the flu with no name needed.

**In one line:** count distinct sensitive values per group - l=1 leaks the secret even at k=4, l>=2 does not.

**What you'll see:** Prints class (02,1980s) with l=1 -> LEAKS (homogeneity attack) and class (10,1970s) with l=4 -> OK, then the explanation that both are k-anonymous but only one is l-diverse.

## 5 - Buy the guarantee that survives any side data: differential privacy

k-anonymity and l-diversity protect a released table but can still fall to an attacker with enough outside data. **Differential privacy** promises that whether or not any single person is in the dataset changes any output by at most a factor of e^epsilon. The **Laplace mechanism** adds noise scaled to sensitivity/epsilon; epsilon is a privacy **budget** - smaller means more noise and stronger privacy, and epsilons compose across queries.

In [ ]:
# DIFFERENTIAL PRIVACY: the strongest guarantee. Add calibrated noise so one person joining or leaving barely changes any
# released answer - formally, the odds of any output change by at most e^epsilon. Smaller epsilon = more noise = more privacy.
import math
true_count = 42                                  # e.g. patients with a condition; hidden behind noise before release
sensitivity = 1                                  # one person changes the true count by at most 1
print("Releasing a count with the Laplace mechanism (noise ~ Laplace(0, sensitivity/epsilon)):")
for eps, drawn_noise in [(1.0, 1.3), (0.1, 8.0)]:  # drawn_noise = one illustrative fixed sample (real DP draws it randomly)
    scale = sensitivity / eps                    # bigger scale = more noise
    released = true_count + drawn_noise
    print("  epsilon={:.1f}: noise scale=sens/eps={:>4.1f}, privacy bound e^eps={:.2f}, released ~= {:.0f}".format(eps, scale, math.e ** eps, released))
print()
print("epsilon is a privacy BUDGET: e^0.1=1.11 (strong, noisy) vs e^1.0=2.72 (weak, accurate). And it COMPOSES -")
print("two queries at epsilon=0.5 each spend epsilon=1.0 total. DP is the one guarantee that survives an attacker with side data.")

# Output:
# Releasing a count with the Laplace mechanism (noise ~ Laplace(0, sensitivity/epsilon)):
#   epsilon=1.0: noise scale=sens/eps= 1.0, privacy bound e^eps=2.72, released ~= 43
#   epsilon=0.1: noise scale=sens/eps=10.0, privacy bound e^eps=1.11, released ~= 50
#
# epsilon is a privacy BUDGET: e^0.1=1.11 (strong, noisy) vs e^1.0=2.72 (weak, accurate). And it COMPOSES -
# two queries at epsilon=0.5 each spend epsilon=1.0 total. DP is the one guarantee that survives an attacker with side data.

**Explanation**

A demonstration of the Laplace mechanism at two privacy budgets, using one fixed illustrative noise sample instead of a real random draw (for reproducibility). It shows how lowering epsilon multiplies the noise scale and tightens the e^epsilon guarantee, trading accuracy for privacy.

**How the code works - step by step**
- **`import math`** - only for `math.e ** eps` to display the privacy bound.
- **`true_count`, `sensitivity`** - the real answer (42) and how much one person can move it (1, for a count).
- **The loop over `(eps, drawn_noise)`** - for epsilon 1.0 and 0.1, computes `scale = sensitivity / eps`, the bound `e^eps`, and the released noisy value `true_count + drawn_noise`.
- **`drawn_noise`** - a fixed stand-in for the random Laplace draw so the output is deterministic (the comment says real DP draws it randomly).
- **The printout** - contrasts e^0.1=1.11 (strong, noisy) with e^1.0=2.72 (weak, accurate) and notes epsilons compose: two queries at 0.5 spend 1.0.

**In one line:** smaller epsilon -> bigger noise scale (sensitivity/epsilon) -> tighter e^epsilon guarantee, and the budget composes across queries.

**What you'll see:** Prints two lines: epsilon=1.0 with scale 1.0, bound 2.72, released ~=43; and epsilon=0.1 with scale 10.0, bound 1.11, released ~=50 - then the budget/composition note.

## 6 - Test whether a trained model leaks who was in it

Even a model that never emits raw data can leak **membership** - whether a specific record was in its training set. Models overfit, so training examples get **lower loss** than unseen ones, and an attacker just thresholds the loss to guess. A wide train/holdout loss gap is overfitting **and** a privacy leak; membership alone can be sensitive ('was in the HIV study'). Defend with regularization, early stopping, and DP-SGD.

In [ ]:
# MEMBERSHIP INFERENCE: even a model that never emits raw data can leak WHO was in its training set. A model fits its
# training examples too well, so they get LOWER loss than unseen ones - an attacker thresholds the loss to guess membership.
member_loss = [0.10, 0.18, 0.22, 0.15, 0.30]        # examples the model trained on -> memorized -> low loss
holdout_loss = [0.70, 0.55, 0.82, 0.60, 0.75]       # unseen examples -> higher loss
THRESH = 0.5                                          # attacker guesses "was a member" if loss < THRESH
tp = sum(1 for x in member_loss if x < THRESH)       # members correctly flagged
tn = sum(1 for x in holdout_loss if x >= THRESH)     # non-members correctly cleared
attack_acc = (tp + tn) / (len(member_loss) + len(holdout_loss))
print("Membership-inference attack (threshold the model's per-example loss):")
print("  members flagged   : {}/{}".format(tp, len(member_loss)))
print("  non-members cleared: {}/{}".format(tn, len(holdout_loss)))
print("  attack accuracy   : {:.0%}  (50% = the attacker learns nothing; higher = the model leaks membership)".format(attack_acc))
print()
print("A wide train/holdout loss gap is overfitting AND a privacy leak - membership alone can be sensitive ('was in the HIV study').")
print("The defenses are the same as for generalization: regularization, early stopping, and training with differential privacy (DP-SGD).")

# Output:
# Membership-inference attack (threshold the model's per-example loss):
#   members flagged   : 5/5
#   non-members cleared: 5/5
#   attack accuracy   : 100%  (50% = the attacker learns nothing; higher = the model leaks membership)
#
# A wide train/holdout loss gap is overfitting AND a privacy leak - membership alone can be sensitive ('was in the HIV study').
# The defenses are the same as for generalization: regularization, early stopping, and training with differential privacy (DP-SGD).

**Explanation**

A loss-threshold membership-inference attack run over five training and five held-out records. It thresholds each record's per-example loss and scores how accurately the attacker separates members from non-members, showing that a clean loss gap makes the attack perfect (100 percent) and that 50 percent would mean no leak.

**How the code works - step by step**
- **`member_loss` / `holdout_loss`** - five low training losses (0.10-0.30) and five higher held-out losses (0.55-0.82).
- **`THRESH`** - the attacker's rule: loss below 0.5 is guessed 'was a member'.
- **`tp`** - members correctly flagged (loss < THRESH); **`tn`** - non-members correctly cleared (loss >= THRESH).
- **`attack_acc`** - (tp + tn) over all ten records; 50 percent means the attacker learns nothing, higher means the model leaks.
- **The printout** - reports the flagged/cleared counts and accuracy, then names the defenses that also fix overfitting.

**In one line:** threshold the per-example loss - a wide member/holdout gap separates them perfectly, so membership leaks at 100 percent.

**What you'll see:** Prints members flagged 5/5, non-members cleared 5/5, attack accuracy 100%, then the note that the loss gap is both overfitting and a leak and that DP-SGD is the formal defense.

## 7 - Gate the pipeline and confront the right to erasure

Privacy is an engineering **gate**, not a promise. Minimize what you collect, redact or pseudonymize before the model call, never log raw PII, encrypt, and honor deletion (GDPR Article 17). Erasing a person from a **trained** model - 'unlearning' - is hard because retraining is costly, so the durable answer is to minimize and pseudonymize up front so a later erasure request stays tractable.

In [ ]:
# THE PII PIPELINE + RIGHT TO ERASURE: privacy is an engineering gate, not a promise. Minimize what you collect, redact
# before the model sees it, never log raw PII, encrypt, and honor deletion (GDPR Art. 17 / CCPA). Erasing from a TRAINED model is hard.
CONTROLS = {                                          # each must hold for the pipeline to ship
    "collect only what is needed (data minimization)": True,
    "redact / pseudonymize PII before the model call": True,
    "do not log raw prompts or PII": False,           # UNMET - raw prompts are being logged
    "honor data-subject erasure (GDPR Art. 17)": True,
    "encrypt PII in transit and at rest": True}
unmet = [name for name, ok in CONTROLS.items() if not ok]
print("PRIVACY PIPELINE GATE (before a feature that touches personal data ships):")
for name, ok in CONTROLS.items():
    print("  [{}] {}".format("x" if ok else " ", name))
print()
print("gate: {}".format("PASS" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("Right-to-erasure is the hardest: deleting a row from a database is easy, but 'unlearning' a person from a trained model")
print("is not - retraining is costly. The practical answer is to minimize and pseudonymize UP FRONT so erasure stays tractable.")

# Output:
# PRIVACY PIPELINE GATE (before a feature that touches personal data ships):
#   [x] collect only what is needed (data minimization)
#   [x] redact / pseudonymize PII before the model call
#   [ ] do not log raw prompts or PII
#   [x] honor data-subject erasure (GDPR Art. 17)
#   [x] encrypt PII in transit and at rest
#
# gate: BLOCK - 1 unmet:
#    - do not log raw prompts or PII
# Right-to-erasure is the hardest: deleting a row from a database is easy, but 'unlearning' a person from a trained model
# is not - retraining is costly. The practical answer is to minimize and pseudonymize UP FRONT so erasure stays tractable.

**Explanation**

A gate harness that fails safe. It checks five required privacy controls and blocks the release if any one is unmet - here 'do not log raw prompts or PII' is False, so the whole feature is blocked - and closes on why the hardest control, erasure, argues for minimizing at ingestion.

**How the code works - step by step**
- **`CONTROLS`** - a dictionary of the five gate controls mapped to True/False; one (do-not-log-raw-PII) is deliberately False.
- **`unmet`** - the list of controls whose value is False.
- **The checklist loop** - prints each control with `[x]` or `[ ]` for pass/fail.
- **The gate verdict** - PASS only if `unmet` is empty, otherwise BLOCK with the count and the offending controls listed.
- **The erasure note** - explains that deleting a database row is easy but unlearning a trained model is not, so minimize and pseudonymize up front.

**In one line:** one unmet control blocks the whole release - the gate fails safe, and erasure is why you minimize at ingestion.

**What you'll see:** Prints the five-item checklist with one unchecked box, then 'gate: BLOCK - 1 unmet: do not log raw prompts or PII', followed by the right-to-erasure explanation.

Across seven small, keyless blocks you turned "privacy" from a vibe into a set of numbers you can put on a release: classify the fields, measure how re-identifiable each record is, raise k until people hide in a crowd, add l-diversity so the crowd does not share one secret, buy the differential-privacy guarantee that survives an attacker with unlimited side data, test a trained model for membership leakage, and gate the whole pipeline so raw PII never comes to rest. The recurring lesson is that anonymity is a property of your data plus everything an attacker can link to it, and that the cheapest privacy control is minimizing and pseudonymizing at ingestion. Next up is 15.5 (synthetic likeness and deepfakes) and 15.6 (the GDPR / EU AI Act obligations that turn these engineering measures into law).